# VideoRAG Colab Evaluation
- Mount Google Drive để lưu/đọc dữ liệu nhanh.
- Copy `longervideos.tar` và `all_answers.zip` trực tiếp từ Drive (tùy chọn fallback gdown nếu muốn dùng FILE_ID).
- Giải nén dữ liệu.
- (Tùy chọn) ingest vài video mẫu để warm up storage.
- Chạy so sánh pipeline mới (ielts_rag) với baseline.

In [ ]:
# Thiết lập tham số
REPO_URL = "https://github.com/thang3092004/IELTS-RAG.git"
BRANCH = "main"
# Đặt đường dẫn file nén trên Drive; để trống nếu muốn dùng gdown
DRIVE_LONGERVIDEOS = "/content/drive/MyDrive/path/to/longervideos.tar"
DRIVE_ALL_ANSWERS = "/content/drive/MyDrive/path/to/all_answers.zip"
USE_GDOWN = False  # True nếu muốn tải bằng gdown qua FILE_ID
FILE_ID_LONGERVIDEOS = ""  # Drive ID của longervideos.tar (nếu USE_GDOWN=True)
FILE_ID_ALL_ANSWERS = ""    # Drive ID của all_answers.zip (nếu USE_GDOWN=True)
OPENAI_API_KEY = ""  # nhập nếu dùng OpenAI
USE_INGEST = True      # True để ingest mẫu; False nếu đã có working_dir sẵn
DATASET_ID = 17
QUESTION_ID = 1

In [ ]:
# Mount Google Drive để lưu persistent working_dir
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
WORKDIR = Path('/content/drive/MyDrive/videorag_workdir')
WORKDIR.mkdir(parents=True, exist_ok=True)
WORKDIR

In [ ]:
# Cài gói cần thiết
!pip install -q gdown tiktoken networkx transformers openai sentencepiece

In [ ]:
# Clone repo
!rm -rf IELTS-RAG
!git clone --branch $BRANCH $REPO_URL
%cd IELTS-RAG

In [ ]:
# Lấy longervideos.tar và all_answers.zip
import os
from pathlib import Path

if USE_GDOWN:
    if not FILE_ID_LONGERVIDEOS:
        raise ValueError('Điền FILE_ID_LONGERVIDEOS từ link Drive hoặc đặt USE_GDOWN=False để copy từ Drive đã mount')
    if not FILE_ID_ALL_ANSWERS:
        raise ValueError('Điền FILE_ID_ALL_ANSWERS từ link Drive hoặc đặt USE_GDOWN=False để copy từ Drive đã mount')
    !gdown --id $FILE_ID_LONGERVIDEOS -O longervideos.tar
    !gdown --id $FILE_ID_ALL_ANSWERS -O all_answers.zip
else:
    drive_video = Path(DRIVE_LONGERVIDEOS)
    drive_answers = Path(DRIVE_ALL_ANSWERS)
    if not drive_video.exists():
        raise FileNotFoundError(f"Không thấy DRIVE_LONGERVIDEOS: {drive_video}")
    if not drive_answers.exists():
        raise FileNotFoundError(f"Không thấy DRIVE_ALL_ANSWERS: {drive_answers}")
    !cp "$DRIVE_LONGERVIDEOS" longervideos.tar
    !cp "$DRIVE_ALL_ANSWERS" all_answers.zip

!ls -lh longervideos.tar all_answers.zip

In [ ]:
# Giải nén dữ liệu
!tar -xf longervideos.tar
!unzip -q all_answers.zip -d reproduce/all_answers
!ls longervideos | head

In [ ]:
# Đặt API key (nếu có)
import os
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [ ]:
# (Tùy chọn) Ingest một vài video đầu; có thể resume, video đã xử lý sẽ bị skip
from pathlib import Path
import glob, json
from videorag.videorag import VideoRAG

workdir = WORKDIR  # dùng thư mục trên Drive để bền sau khi ngắt phiên
workdir.mkdir(exist_ok=True)

if USE_INGEST:
    ds = json.loads(Path('longervideos/dataset.json').read_text())
    desc = ds[str(DATASET_ID)][0]['description']
    question = ds[str(DATASET_ID)][0]['questions'][QUESTION_ID]['question']
    video_paths = sorted(glob.glob(f'longervideos/{DATASET_ID}-{desc}/videos/*'))
    sample_videos = video_paths[:2]
    print(f'Using {len(sample_videos)} videos for ingest')
    vr = VideoRAG(working_dir=str(workdir))
    # insert_video sẽ bỏ qua video đã có trong storage, cho phép resume
    vr.insert_video(sample_videos)
else:
    print('Skipping ingest; cần working_dir sẵn nếu muốn query')

In [ ]:
# (Tùy chọn) Tạo checkpoint working_dir lên Drive (nén nhanh)
import datetime, subprocess
timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
archive_path = WORKDIR.parent / f'videorag_workdir_{timestamp}.tar.gz'
!tar -czf $archive_path -C $WORKDIR .. /$(basename $WORKDIR)
archive_path

In [ ]:
# Chạy so sánh pipeline mới với baseline
from pathlib import Path
import json
from difflib import SequenceMatcher
from videorag.base import QueryParam
from videorag.videorag import VideoRAG

ds = json.loads(Path('longervideos/dataset.json').read_text())
desc = ds[str(DATASET_ID)][0]['description']
question = ds[str(DATASET_ID)][0]['questions'][QUESTION_ID]['question']
baseline_path = Path(f'reproduce/all_answers/{DATASET_ID}-{desc}/answers-videorag/answer_{QUESTION_ID}.md')
baseline = baseline_path.read_text(encoding='utf-8').strip() if baseline_path.exists() else '<no baseline>'

vr = VideoRAG(working_dir=str(workdir), always_create_working_dir=False)
param = QueryParam()
param.mode = 'ielts_rag'
param.top_k = 20
resp = vr.query(question, param=param)

if isinstance(resp, dict):
    model_answer = resp.get('answer', resp)
    rationale = resp.get('rationale', '')
    citations = resp.get('citations', [])
else:
    model_answer = resp
    rationale = ''
    citations = []

sim = SequenceMatcher(None, baseline, str(model_answer)).ratio() if baseline != '<no baseline>' else 0.0
print('Dataset:', desc)
print('Question:', question)
print('\n=== Baseline ===\n', baseline)
print('\n=== Model (ielts_rag) ===\n', model_answer)
print('\nRationale ===\n', rationale)
print('\nCitations ===\n', citations)
print(f'\nSimilarity: {sim:.3f}')

## Ghi chú
- Ưu tiên copy từ Drive đã mount để nhanh hơn; chỉ dùng gdown khi cần tải bằng FILE_ID.
- Giảm số video ingest nếu thời gian lâu; insert_video sẽ bỏ qua video đã có trong storage nên có thể resume nếu bị ngắt.
- WORKDIR đặt trên Drive nên phiên bị dừng vẫn giữ được dữ liệu. Có thể dùng cell checkpoint để nén dự phòng.
- Nếu đã có working_dir khác, đặt USE_INGEST=False và trỏ WORKDIR tới thư mục đó.
- Nếu thiếu mô hình cục bộ, notebook giả định dùng OpenAI; cần cấu hình thêm nếu muốn chạy local models.